# NYSC Survival · real streets and landmarks from OpenStreetMap
Runtime → **Run all**. About 10–15 minutes, no key, nothing to pay, and no busy map servers:
it downloads the whole Nigeria map file once (~600 MB, from Geofabrik) and cuts out each state's town centre itself.
At the end it downloads `nysc_osm.zip` → send it to Claude.
Data © OpenStreetMap contributors, ODbL. The game credits it.

In [ ]:
# 1 · tools + the Nigeria map file
!apt-get -qq install -y osmium-tool > /dev/null
!wget -q -nc -O /content/ng.osm.pbf https://download.geofabrik.de/africa/nigeria-latest.osm.pbf
import json, math, time, os, requests, shutil, subprocess
print('map file:', os.path.getsize('/content/ng.osm.pbf') // 2**20, 'MB')
AREAS = [["AB", "Umuahia, Abia State"], ["AD", "Jimeta, Yola, Adamawa State"], ["AK", "Uyo, Akwa Ibom State"], ["AN", "Awka, Anambra State"], ["BA", "Bauchi, Bauchi State"], ["BY", "Yenagoa, Bayelsa State"], ["BE", "Makurdi, Benue State"], ["BO", "Maiduguri, Borno State"], ["CR", "Calabar, Cross River State"], ["DT", "Asaba, Delta State"], ["EB", "Abakaliki, Ebonyi State"], ["ED", "Ring Road, Benin City, Edo State"], ["EK", "Ado-Ekiti, Ekiti State"], ["EN", "Okpara Square, Enugu, Enugu State"], ["FC", "Wuse, Abuja, FCT"], ["GB", "Gombe, Gombe State"], ["IM", "Owerri, Imo State"], ["JG", "Dutse, Jigawa State"], ["KD", "Kaduna, Kaduna State"], ["KN", "Kano, Kano State"], ["KT", "Katsina, Katsina State"], ["KB", "Birnin Kebbi, Kebbi State"], ["KG", "Lokoja, Kogi State"], ["KW", "Ilorin, Kwara State"], ["LA", "Ikeja, Lagos State"], ["NR", "Lafia, Nasarawa State"], ["NG", "Minna, Niger State"], ["OG", "Abeokuta, Ogun State"], ["OD", "Akure, Ondo State"], ["OS", "Osogbo, Osun State"], ["OY", "Dugbe, Ibadan, Oyo State"], ["PL", "Jos, Plateau State"], ["RV", "Port Harcourt, Rivers State"], ["SO", "Sokoto, Sokoto State"], ["TR", "Jalingo, Taraba State"], ["YB", "Damaturu, Yobe State"], ["ZA", "Gusau, Zamfara State"]]
UA = {'User-Agent': 'NYSC-Survival-game/1.0 (github.com/Auspicious14/nysc-survival)'}
R, RL = 600, 15000
OUT = '/content/nysc_osm'; os.makedirs(OUT, exist_ok=True)

In [ ]:
# 2 · find each town centre (OpenStreetMap's own search, 1 per second)
CENTRES = {}
for code_, q in AREAS:
    for attempt in range(3):
        try:
            j = requests.get('https://nominatim.openstreetmap.org/search', params={'q': q + ', Nigeria', 'format': 'json', 'limit': 1}, headers=UA, timeout=60).json()
            if j: CENTRES[code_] = (float(j[0]['lat']), float(j[0]['lon'])); break
        except Exception as e: print('  retry', code_, str(e)[:60])
        time.sleep(3)
    time.sleep(1.2)
    print(code_, q, CENTRES.get(code_, 'NOT FOUND'))

In [ ]:
# 3 · cut each town out of the Nigeria file in one pass, and pull every Wikidata-linked place for landmarks
def bbox(lat, lon, r):
    dlat = r / 111320; dlon = r / (111320 * math.cos(math.radians(lat)))
    return [lon - dlon, lat - dlat, lon + dlon, lat + dlat]
os.makedirs('/content/ex', exist_ok=True)
cfg = {'directory': '/content/ex', 'extracts': [{'output': f'{c}.osm.pbf', 'bbox': bbox(*CENTRES[c], R + 80)} for c in CENTRES]}
json.dump(cfg, open('/content/ex.json', 'w'))
subprocess.run(['osmium', 'extract', '-c', '/content/ex.json', '/content/ng.osm.pbf', '--overwrite'], check=True)
subprocess.run(['osmium', 'tags-filter', '/content/ng.osm.pbf', 'nwr/wikidata', '-o', '/content/wd.osm.pbf', '--overwrite'], check=True)
subprocess.run(['osmium', 'export', '/content/wd.osm.pbf', '-f', 'geojsonseq', '-o', '/content/wd.geojsonseq', '--overwrite', '-x', 'print_record_separator=false'], check=True)
print('cut', len(os.listdir('/content/ex')), 'towns')

In [ ]:
# 4 · turn each town into the small format the game reads
def feats(path):
    for line in open(path, encoding='utf-8'):
        line = line.strip().lstrip('\x1e')
        if line: yield json.loads(line)
def rings(g):
    t = g['type']; c = g['coordinates']
    if t == 'Point': return [[c]]
    if t == 'LineString': return [c]
    if t == 'Polygon': return [c[0]]
    if t == 'MultiPolygon': return [p[0] for p in c]
    if t == 'MultiLineString': return c
    return []
KEEP_ROAD = {'motorway','trunk','primary','secondary','tertiary','unclassified','residential','service','living_street','pedestrian','footway','path','track','motorway_link','trunk_link','primary_link','secondary_link','tertiary_link'}
WD = list(feats('/content/wd.geojsonseq'))
for code_, (lat, lon) in CENTRES.items():
    k = 111320 * math.cos(math.radians(lat))
    P = lambda la, lo: (round((lo - lon) * k, 1), round((la - lat) * 111320, 1))
    src = f'/content/ex/{code_}.osm.pbf'; gj = f'/content/ex/{code_}.geojsonseq'
    subprocess.run(['osmium', 'export', src, '-f', 'geojsonseq', '-o', gj, '--overwrite', '-x', 'print_record_separator=false'], check=True)
    roads, bld, poi, green, water = [], [], [], [], []
    for f in feats(gj):
        t = f['properties']; nm = t.get('name', ''); g = f['geometry']
        if g['type'] == 'Point':
            kk = t.get('amenity') or t.get('shop') or t.get('office') or t.get('tourism') or ('bus_stop' if t.get('highway') == 'bus_stop' else '') or ('place:' + t['place'] if 'place' in t else '')
            if kk: x, y = P(g['coordinates'][1], g['coordinates'][0]); poi.append([kk, nm, x, y])
            continue
        for ring in rings(g):
            pts = [v for lo, la in ring for v in P(la, lo)]
            if not pts or min(math.hypot(pts[i], pts[i + 1]) for i in range(0, len(pts), 2)) > R + 60: continue
            if t.get('highway') in KEEP_ROAD:
                ln = str(t.get('lanes', '0')); roads.append([t['highway'], nm, int(ln) if ln.isdigit() else 0, 1 if t.get('oneway') == 'yes' else 0, pts])
            elif 'building' in t:
                lv = str(t.get('building:levels', '')); lv = int(float(lv)) if lv.replace('.', '', 1).isdigit() else 0
                bld.append([t['building'], nm, lv, t.get('amenity') or t.get('shop') or t.get('religion') or '', pts])
            elif t.get('natural') == 'water' or 'waterway' in t: water.append([t.get('waterway') or 'water', nm, pts])
            elif t.get('landuse') in ('grass', 'forest', 'meadow', 'farmland', 'recreation_ground', 'cemetery') or t.get('leisure') in ('park', 'pitch', 'garden', 'stadium') or t.get('natural') in ('wood', 'scrub', 'grassland'):
                green.append([t.get('leisure') or t.get('landuse') or t.get('natural'), nm, pts])
            elif 'amenity' in t and nm:
                poi.append([t['amenity'], nm, round(sum(pts[0::2]) / (len(pts) // 2), 1), round(sum(pts[1::2]) / (len(pts) // 2), 1)])
    lm = []
    for f in WD:
        t = f['properties']
        if not t.get('name'): continue
        kind = next((f'{kk}={t[kk]}' for kk in ('tourism', 'historic', 'amenity', 'leisure', 'man_made', 'natural', 'bridge', 'aeroway', 'office', 'building', 'railway') if kk in t), '')
        if not kind or kind.startswith(('amenity=school', 'amenity=bank', 'amenity=fuel', 'amenity=restaurant')): continue
        r = rings(f['geometry'])
        if not r: continue
        pts = [p for ring in r for p in ring]
        cla = sum(p[1] for p in pts) / len(pts); clo = sum(p[0] for p in pts) / len(pts)
        x, y = P(cla, clo); d = math.hypot(x, y)
        if d > RL: continue
        lm.append([kind, t['name'], round(d), round(math.degrees(math.atan2(x, y)) % 360), t.get('religion', ''), t.get('ele', ''), t.get('height', ''), t.get('wikidata', ''), x, y])
    out = {'id': code_, 'q': dict(AREAS)[code_], 'center': [lat, lon], 'r': R, 'roads': roads, 'bld': bld, 'poi': poi, 'green': green, 'water': water, 'landmarks': sorted(lm, key=lambda v: v[2])}
    json.dump(out, open(f'{OUT}/{code_}.json', 'w'), separators=(',', ':'), ensure_ascii=False)
    print(f"{code_}: {len(roads)} roads · {len(bld)} buildings · {len(poi)} places · {len(lm)} landmarks · {os.path.getsize(f'{OUT}/{code_}.json') // 1024} KB")
shutil.make_archive('/content/nysc_osm', 'zip', OUT)
from google.colab import files; files.download('/content/nysc_osm.zip')